In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

pd.set_option("display.max_columns", None)

FIGURES_DIR = Path("../results/figures/fase2")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [21]:
def confidence_interval(series):
    """
    Calcula média, desvio padrão, erro padrão e
    intervalo de confiança de 95%.

    Usa aproximação normal (n = 30).
    """
    n = len(series)

    mean = series.mean()
    std = series.std(ddof=1)

    se = std / np.sqrt(n)

    ci95 = 1.96 * se

    return pd.Series({
        "mean": mean,
        "std": std,
        "ci95_lower": mean - ci95,
        "ci95_upper": mean + ci95,
        "ci95": ci95
    })

In [2]:
validation = pd.read_csv("../results/simulation/validation_metrics.csv")
jitter = pd.read_csv("../results/simulation/jitter_metrics.csv")
window = pd.read_csv("../results/simulation/window_metrics.csv")
file_size = pd.read_csv("../results/simulation/file_size_metrics.csv")
stress = pd.read_csv("../results/simulation/stress_metrics.csv")

validation.head()

,experiment,run,scenario,file_size_bytes,chunk_size_bytes,window_size,timeout_s,max_retries,delay_mean_ms,delay_std_ms,loss_probability,original_data_packets,total_data_packets,retransmitted_packets,ack_packets_sent,ack_packets_received,data_packets_lost,ack_packets_lost,useful_data_bytes,data_bytes_sent,ack_bytes_sent,total_bytes_sent,duration_s,throughput_bps,network_throughput_bps,mean_rtt_s,efficiency_ratio
0,validation,1,A,524288,1024,4,1.0,100,10,0.0,0.0,512,512,0,512,512,0,0,524288,524288,20480,544768,2.56,204800.000000,212800.000000,0.020000,0.500000
1,validation,1,B,524288,1024,4,1.0,100,50,0.0,0.1,512,676,164,626,570,50,56,524288,692224,25040,717264,55.90,9379.033989,12831.198569,0.133684,0.393241
2,validation,1,C,524288,1024,4,1.0,100,100,0.0,0.2,512,980,468,781,616,199,165,524288,1003520,31240,1034760,153.00,3426.718954,6763.137255,0.332792,0.290744
3,validation,2,A,524288,1024,4,1.0,100,10,0.0,0.0,512,512,0,512,512,0,0,524288,524288,20480,544768,2.56,204800.000000,212800.000000,0.020000,0.500000
4,validation,2,B,524288,1024,4,1.0,100,50,0.0,0.1,512,741,229,660,585,81,75,524288,758784,26400,785184,73.70,7113.812754,10653.785617,0.137436,0.365453


In [3]:
print("Validation:", validation.shape)
print("Jitter:", jitter.shape)
print("Window:", window.shape)
print("File size:", file_size.shape)
print("Stress:", stress.shape)

Validation: (90, 27)
Jitter: (450, 27)
Window: (540, 27)
File size: (450, 27)
Stress: (30, 27)


In [4]:
validation_summary = (
    validation
    .groupby("scenario")
    .agg(
        duration_mean=("duration_s", "mean"),
        duration_std=("duration_s", "std"),
        throughput_mean=("throughput_bps", "mean"),
        throughput_std=("throughput_bps", "std"),
        retrans_mean=("retransmitted_packets", "mean"),
        retrans_std=("retransmitted_packets", "std"),
        rtt_mean=("mean_rtt_s", "mean"),
        rtt_std=("mean_rtt_s", "std"),
        efficiency_mean=("efficiency_ratio", "mean"),
        efficiency_std=("efficiency_ratio", "std")
    )
    .reset_index()
)

validation_summary

,scenario,duration_mean,duration_std,throughput_mean,throughput_std,retrans_mean,retrans_std,rtt_mean,rtt_std,efficiency_mean,efficiency_std
0,A,2.560000,0.000000,204800.000000,0.000000,0.000000,0.000000,0.020000,0.000000,0.500000,0.000000
1,B,74.036667,6.953564,7143.901655,696.778355,233.233333,26.313342,0.144541,0.012781,0.362252,0.011153
2,C,164.140000,12.413024,3211.598608,240.157704,506.800000,45.932034,0.341084,0.023620,0.279345,0.011297


In [5]:
fig = px.bar(
    validation_summary,
    x="scenario",
    y="throughput_mean",
    error_y="throughput_std",
    title="Fase 2 - Throughput médio simulado por cenário",
    labels={
        "scenario": "Cenário",
        "throughput_mean": "Throughput médio simulado (B/s)"
    }
)

fig.write_image(
    FIGURES_DIR / "fase2_throughput_simulado.png",
    width=1400,
    height=800
)

fig.show()

In [6]:
fig = px.bar(
    validation_summary,
    x="scenario",
    y="retrans_mean",
    error_y="retrans_std",
    title="Fase 2 - Retransmissões médias simuladas por cenário",
    labels={
        "scenario": "Cenário",
        "retrans_mean": "Retransmissões médias"
    }
)

fig.write_image(
    FIGURES_DIR / "fase2_retransmissoes_simuladas.png",
    width=1400,
    height=800
)

fig.show()

In [7]:
fig = px.bar(
    validation_summary,
    x="scenario",
    y="duration_mean",
    error_y="duration_std",
    title="Fase 2 - Duração média simulada por cenário",
    labels={
        "scenario": "Cenário",
        "duration_mean": "Duração média simulada (s)"
    }
)

fig.write_image(
    FIGURES_DIR / "fase2_duracao_simulada.png",
    width=1400,
    height=800
)

fig.show()

## Comparação dos resultados da Fase 01 com a Fase 02

In [11]:
real = pd.read_csv("../results/app_metrics_test_512kb.csv")
real.head()

,run,scenario,protocol,file,delay_ms,loss_percent,status,bytes_expected,bytes_received,delivery_rate,elapsed_time_s,throughput_bps,retransmissions,pcap_file,network_csv_file
0,1,A,TCP,test_512kb.bin,10,0,success,524288,524288,1.0,0.0527,9943402.59,NaN,captures/tcp_scenario_a_run_1.pcap,captures/tcp_scenario_a_run_1.csv
1,1,A,RUDP,test_512kb.bin,10,0,success,524288,524288,1.0,1.5590,336293.56,0.0,captures/rudp_scenario_a_run_1.pcap,captures/rudp_scenario_a_run_1.csv
2,1,B,TCP,test_512kb.bin,50,10,success,524288,524288,1.0,1.7639,297228.73,NaN,captures/tcp_scenario_b_run_1.pcap,captures/tcp_scenario_b_run_1.csv
3,1,B,RUDP,test_512kb.bin,50,10,success,524288,524288,1.0,58.1541,9015.49,191.0,captures/rudp_scenario_b_run_1.pcap,captures/rudp_scenario_b_run_1.csv
4,1,C,TCP,test_512kb.bin,100,20,success,524288,524288,1.0,14.8159,35386.92,NaN,captures/tcp_scenario_c_run_1.pcap,captures/tcp_scenario_c_run_1.csv


In [12]:
real_summary = (
    real
    .groupby(["scenario","protocol"])
    .agg(
        throughput=("throughput_bps","mean"),
        throughput_std=("throughput_bps","std"),

        duration=("elapsed_time_s","mean"),
        duration_std=("elapsed_time_s","std"),

        retrans=("retransmissions","mean")
    )
    .reset_index()
)

real_summary

,scenario,protocol,throughput,throughput_std,duration,duration_std,retrans
0,A,RUDP,337097.746,1.322415e+03,1.55530,0.006085,0.0
1,A,TCP,9346582.288,1.225946e+06,0.05690,0.007770,NaN
2,B,RUDP,8426.508,9.021284e+02,62.79922,6.797251,208.0
3,B,TCP,442415.776,5.633685e+05,2.87928,2.480990,NaN
4,C,RUDP,3311.390,3.053809e+02,159.40330,14.603814,515.0
5,C,TCP,31874.016,7.229210e+03,17.12396,3.789265,NaN


In [13]:
sim_summary = (
    validation
    .groupby("scenario")
    .agg(
        throughput=("throughput_bps","mean"),
        throughput_std=("throughput_bps","std"),

        duration=("duration_s","mean"),
        duration_std=("duration_s","std"),

        retrans=("retransmitted_packets","mean")
    )
    .reset_index()
)

sim_summary["protocol"] = "RUDP (Sim)"

In [14]:
real_summary["protocol"] = real_summary["protocol"].replace({
    "RUDP":"RUDP (Real)",
    "TCP":"TCP"
})

comparison = pd.concat([
    real_summary,
    sim_summary
], ignore_index=True)

comparison

,scenario,protocol,throughput,throughput_std,duration,duration_std,retrans
0,A,RUDP (Real),3.370977e+05,1.322415e+03,1.555300,0.006085,0.000000
1,A,TCP,9.346582e+06,1.225946e+06,0.056900,0.007770,NaN
2,B,RUDP (Real),8.426508e+03,9.021284e+02,62.799220,6.797251,208.000000
3,B,TCP,4.424158e+05,5.633685e+05,2.879280,2.480990,NaN
4,C,RUDP (Real),3.311390e+03,3.053809e+02,159.403300,14.603814,515.000000
5,C,TCP,3.187402e+04,7.229210e+03,17.123960,3.789265,NaN
6,A,RUDP (Sim),2.048000e+05,0.000000e+00,2.560000,0.000000,0.000000
7,B,RUDP (Sim),7.143902e+03,6.967784e+02,74.036667,6.953564,233.233333
8,C,RUDP (Sim),3.211599e+03,2.401577e+02,164.140000,12.413024,506.800000


### Figs

In [15]:
fig = px.bar(
    comparison,
    x="scenario",
    y="throughput",
    color="protocol",
    barmode="group",
    error_y="throughput_std",
    title="Comparação do Throughput"
)

fig.write_image(
    FIGURES_DIR/"comparacao_throughput.png",
    width=1400,
    height=800
)

fig.show()

- Todos os protocolos

In [16]:
fig = px.bar(
    comparison,
    x="scenario",
    y="throughput",
    color="protocol",
    barmode="group",
    error_y="throughput_std",
    title="Comparação do Throughput"
)

fig.update_yaxes(type="log")

fig.write_image(
    FIGURES_DIR/"comparacao_throughput.png",
    width=1400,
    height=800
)

fig.show()

- Apenas R-UDP

In [17]:
comparison_rudp = comparison[
    comparison["protocol"] != "TCP"
]

In [19]:
fig = px.bar(
    comparison_rudp,
    x="scenario",
    y="throughput",
    color="protocol",
    barmode="group",
    error_y="throughput_std",
    title="Validação do simulador (R-UDP)"
)
fig.write_image(
    FIGURES_DIR/"validacao_simulador_rudp.png",
    width=1400,
    height=800
)


fig.show()

- Intervalo de Confiança

In [22]:
throughput_ci = (
    validation
    .groupby("scenario")["throughput_bps"]
    .apply(confidence_interval)
    .unstack()
)

throughput_ci

,mean,std,ci95_lower,ci95_upper,ci95
scenario,,,,,
A,204800.000000,2.960137e-11,204800.000000,204800.000000,1.059271e-11
B,7143.901655,6.967784e+02,6894.562723,7393.240587,2.493389e+02
C,3211.598608,2.401577e+02,3125.659278,3297.537939,8.593933e+01


In [23]:
plot = throughput_ci.reset_index()

fig = px.bar(
    plot,
    x="scenario",
    y="mean",
    error_y="ci95",
    title="Throughput médio da simulação com IC95%",
    labels={
        "scenario": "Cenário",
        "mean": "Throughput (B/s)"
    }
)

fig.write_image(
    FIGURES_DIR / "throughput_ic95.png",
    width=1400,
    height=800
)

fig.show()

- Jitter

In [24]:
jitter_summary = (
    jitter
    .groupby(["delay_std_ms", "scenario"])
    .agg(
        mean_rtt=("mean_rtt_s", "mean"),
        std_rtt=("mean_rtt_s", "std")
    )
    .reset_index()
)

fig = px.line(
    jitter_summary,
    x="delay_std_ms",
    y="mean_rtt",
    color="scenario",
    markers=True,
    error_y="std_rtt",
    title="Impacto do jitter no RTT médio",
    labels={
        "delay_std_ms": "Jitter (ms)",
        "mean_rtt": "RTT médio (s)",
        "scenario": "Cenário"
    }
)

fig.write_image(
    FIGURES_DIR / "impacto_jitter_rtt.png",
    width=1400,
    height=800
)

fig.show()

- Sensibilidade ao tamanho da janela

In [25]:
window_summary = (
    window
    .groupby(["window_size", "scenario"])
    .agg(
        throughput=("throughput_bps", "mean"),
        throughput_std=("throughput_bps", "std")
    )
    .reset_index()
)

fig = px.line(
    window_summary,
    x="window_size",
    y="throughput",
    color="scenario",
    markers=True,
    error_y="throughput_std",
    title="Impacto do tamanho da janela no throughput",
    labels={
        "window_size": "Tamanho da janela",
        "throughput": "Throughput (B/s)",
        "scenario": "Cenário"
    }
)

fig.write_image(
    FIGURES_DIR / "janela_vs_throughput.png",
    width=1400,
    height=800
)

fig.show()

- Curva throughput por tamanho da janela

In [26]:
file_summary = (
    file_size
    .groupby(["file_size_bytes", "scenario"])
    .agg(
        throughput=("throughput_bps", "mean"),
        throughput_std=("throughput_bps", "std")
    )
    .reset_index()
)

file_summary["file_size_mb"] = (
    file_summary["file_size_bytes"] / (1024 * 1024)
)

fig = px.line(
    file_summary,
    x="file_size_mb",
    y="throughput",
    color="scenario",
    markers=True,
    error_y="throughput_std",
    title="Throughput em função do tamanho do arquivo",
    labels={
        "file_size_mb": "Tamanho do arquivo (MB)",
        "throughput": "Throughput (B/s)",
        "scenario": "Cenário"
    }
)

fig.write_image(
    FIGURES_DIR / "arquivo_vs_throughput.png",
    width=1400,
    height=800
)

fig.show()

- Cenario de estresse

In [27]:
stress_metrics = [
    "throughput_bps",
    "duration_s",
    "retransmitted_packets",
    "efficiency_ratio"
]

stress_mean = (
    stress[stress_metrics]
    .mean()
    .reset_index()
)

stress_mean.columns = ["metric", "value"]

fig = px.bar(
    stress_mean,
    x="metric",
    y="value",
    title="Resumo do cenário de estresse (25% de perda)",
    labels={
        "metric": "Métrica",
        "value": "Valor médio"
    }
)

fig.write_image(
    FIGURES_DIR / "cenario_estresse.png",
    width=1400,
    height=800
)

fig.show()

- Analise da eficiencia

In [28]:
eff_summary = (
    validation
    .groupby("scenario")
    .agg(
        efficiency_mean=("efficiency_ratio","mean"),
        efficiency_std=("efficiency_ratio","std")
    )
    .reset_index()
)

eff_summary

,scenario,efficiency_mean,efficiency_std
0,A,0.500000,0.000000
1,B,0.362252,0.011153
2,C,0.279345,0.011297


In [29]:
fig = px.bar(
    eff_summary,
    x="scenario",
    y="efficiency_mean",
    error_y="efficiency_std",
    title="Eficiência média do protocolo R-UDP",
    labels={
        "scenario":"Cenário",
        "efficiency_mean":"Razão Dados / Total de Pacotes"
    }
)

fig.write_image(
    FIGURES_DIR/"eficiencia.png",
    width=1400,
    height=800
)

fig.show()

- Validação do RTT

In [30]:
network = pd.read_csv("../results/network_metrics_512kb.csv")

print(network.columns)

network.head()

Index(['run', 'scenario', 'protocol', 'pcap_file', 'network_packet_count',
       'network_total_bytes', 'network_duration_s', 'network_throughput_bps'],
      dtype='str')


,run,scenario,protocol,pcap_file,network_packet_count,network_total_bytes,network_duration_s,network_throughput_bps
0,1,A,RUDP,captures\rudp_scenario_a_run_1.pcap,1198,719592,4.110436,175064.641934
1,1,A,TCP,captures\tcp_scenario_a_run_1.pcap,164,577386,8.032551,71880.775658
2,1,B,RUDP,captures\rudp_scenario_b_run_1.pcap,1300,742159,58.153333,12762.106011
3,1,B,TCP,captures\tcp_scenario_b_run_1.pcap,407,552104,5.263348,104895.973865
4,1,C,RUDP,captures\rudp_scenario_c_run_1.pcap,1635,928338,152.531194,6086.217355
